# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 8:** 50 useful B1 verbs, in two audio files — **Part A** (verbs 1–25) and **Part B** (verbs 26–50).

For each verb: English meaning → repeat the infinitive 3× (2nd and 3rd slow) → 4 sentences that rotate through tenses (presente, passato prossimo, imperfetto, futuro, condizionale, congiuntivo presente/imperfetto, imperativo) and persons (io, tu, lui/lei, noi, voi, loro, formal Lei). Short review rounds every ~8 verbs and a full review at the end of each part.

| # | verb | # | verb |
|---|---|---|---|
| 1 | *riuscire* | 26 | *convincere* |
| 2 | *sembrare* | 27 | *condividere* |
| 3 | *rendersi conto* | 28 | *evitare* |
| 4 | *accorgersi* | 29 | *consigliare* |
| 5 | *decidere* | 30 | *rimanere* |
| 6 | *scegliere* | 31 | *crescere* |
| 7 | *spiegare* | 32 | *cambiare* |
| 8 | *succedere* | 33 | *smettere* |
| 9 | *dimenticare* | 34 | *continuare* |
| 10 | *ricordarsi* | 35 | *approfittare* |
| 11 | *preoccuparsi* | 36 | *rinunciare* |
| 12 | *lamentarsi* | 37 | *fidarsi* |
| 13 | *permettere* | 38 | *arrabbiarsi* |
| 14 | *promettere* | 39 | *divertirsi* |
| 15 | *risolvere* | 40 | *annoiarsi* |
| 16 | *raggiungere* | 41 | *affrontare* |
| 17 | *ottenere* | 42 | *sostenere* |
| 18 | *mantenere* | 43 | *proporre* |
| 19 | *offrire* | 44 | *discutere* |
| 20 | *chiedere* | 45 | *rompere* |
| 21 | *rispondere* | 46 | *perdere* |
| 22 | *prestare* | 47 | *vincere* |
| 23 | *aspettarsi* | 48 | *dipendere* |
| 24 | *sperare* | 49 | *risparmiare* |
| 25 | *provare* | 50 | *occuparsi* |

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson08_50Verbi_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson scripts (Part A and Part B)

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSONS = {}

LESSONS["A"] = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 8, Part A (B1): 50 useful verbs — verbs 1–25
# Each verb: meaning -> repeat x3 -> 4–5 sentences across tenses & persons.
# "you" = tu unless the prompt says formal (Lei) or plural (voi).
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson eight, part A. Fifty useful verbs.
(narrator) Today you will learn twenty-five verbs. Part B has twenty-five more.
(narrator) For each verb, you will hear the meaning. Then repeat the verb three times. Then you will make sentences in different tenses, and with different people: I, you, he or she, we, you plural, and they.
(narrator) When the prompt says "you", use {tu}. When it says "formal", use {Lei}. When it says "you, plural", use {voi}.
(narrator) Let's begin.

(section) 1 — riuscire
(narrator) Verb one. To manage, to succeed in doing something: {riuscire}.
(repeat x3 slow) riuscire
(narrator) It is irregular. I manage is {riesco}. They manage is {riescono}.
(prompt) Say: I can't manage to sleep. || Non riesco a dormire.
(prompt) Ask a friend: did you manage to finish? || Sei riuscito a finire?
(prompt) Say: she will manage to pass the exam. || Riuscirà a superare l'esame.
(prompt) Say: I hope they manage to arrive on time. || Spero che riescano ad arrivare in tempo.

(section) 2 — sembrare
(narrator) Verb two. To seem: {sembrare}.
(repeat x3 slow) sembrare
(prompt) Say to a friend: you seem tired. || Sembri stanco.
(prompt) Say: it seemed easy. || Sembrava facile.
(prompt) Say: they seem happy. || Sembrano contenti.
(prompt) Say to two friends: it seems to me that you are right. || Mi sembra che abbiate ragione.

(section) 3 — rendersi conto
(narrator) Verb three. To realize: {rendersi conto}. The past participle is {reso}.
(repeat x3 slow) rendersi conto
(prompt) Say: I didn't realize the mistake. || Non mi sono reso conto dell'errore.
(prompt) Ask a friend: do you realize how late it is? || Ti rendi conto di quanto è tardi?
(prompt) Say: we realized the problem too late. || Ci siamo resi conto del problema troppo tardi.
(prompt) Say: one day they will realize it. || Un giorno se ne renderanno conto.

(section) 4 — accorgersi
(narrator) Verb four. To notice: {accorgersi}. The past participle is {accorto}.
(repeat x3 slow) accorgersi
(prompt) Say: I noticed a mistake. || Mi sono accorto di un errore.
(prompt) Ask two friends: did you notice the change? || Vi siete accorti del cambiamento?
(prompt) Say: she doesn't notice anything. || Non si accorge di niente.
(prompt) Say to a friend: if you pay attention, you will notice the difference. || Se stai attento, ti accorgerai della differenza.

(section) 5 — decidere
(narrator) Verb five. To decide: {decidere}. The past participle is {deciso}.
(repeat x3 slow) decidere
(prompt) Say: we decided to leave. || Abbiamo deciso di partire.
(prompt) Ask a friend: have you decided? || Hai deciso?
(prompt) Say: they will decide tomorrow. || Decideranno domani.
(prompt) Say to a friend: it's important that you decide quickly. || È importante che tu decida in fretta.

(section) 6 — scegliere
(narrator) Verb six. To choose: {scegliere}. It is irregular. I choose is {scelgo}. Chosen is {scelto}.
(repeat x3 slow) scegliere
(prompt) Say: I always choose the same thing. || Scelgo sempre la stessa cosa.
(prompt) Ask a friend: which one did you choose? || Quale hai scelto?
(prompt) Tell a friend: choose a number! || Scegli un numero!
(prompt) Say: I think they will choose the University of Bologna. Use the subjunctive. || Penso che scelgano l'Università di Bologna.

(section) 7 — spiegare
(narrator) Verb seven. To explain: {spiegare}.
(repeat x3 slow) spiegare
(prompt) Ask a friend: can you explain it to me? || Me lo puoi spiegare?
(prompt) Say: the professor explained the lesson well. || Il professore ha spiegato bene la lezione.
(prompt) Say to a friend: tomorrow I will explain everything to you. || Domani ti spiegherò tutto.
(prompt) Say, formally: explain it to me, please. || Me lo spieghi, per favore.

(section) 8 — succedere
(narrator) Verb eight. To happen: {succedere}. The past participle is {successo}, and it uses {essere}.
(repeat x3 slow) succedere
(prompt) Ask: what happened? || Che cosa è successo?
(prompt) Say: it happens often. || Succede spesso.
(prompt) Ask: what will happen? || Che cosa succederà?
(prompt) Say: I'm afraid something has happened. || Ho paura che sia successo qualcosa.

(section) Review 1
(narrator) Quick review.
(prompt) Say: I can't manage to sleep. || Non riesco a dormire.
(prompt) Say: it seemed easy. || Sembrava facile.
(prompt) Say: I noticed a mistake. || Mi sono accorto di un errore.
(prompt) Say: we decided to leave. || Abbiamo deciso di partire.
(prompt) Ask a friend: which one did you choose? || Quale hai scelto?

(section) 9 — dimenticare
(narrator) Verb nine. To forget: {dimenticare}.
(repeat x3 slow) dimenticare
(prompt) Say: I forgot the keys. || Ho dimenticato le chiavi.
(prompt) Tell a friend: don't forget! || Non dimenticare!
(prompt) Say: we will never forget this day. || Non dimenticheremo mai questo giorno.
(prompt) Say: he always used to forget my birthday. || Dimenticava sempre il mio compleanno.

(section) 10 — ricordarsi
(narrator) Verb ten. To remember: {ricordarsi}.
(repeat x3 slow) ricordarsi
(prompt) Ask a friend: do you remember me? || Ti ricordi di me?
(prompt) Say: I don't remember his name. || Non mi ricordo il suo nome.
(prompt) Tell a friend: remember to call your mother! || Ricordati di chiamare la mamma!
(prompt) Say: they didn't remember the appointment. || Non si sono ricordati dell'appuntamento.

(section) 11 — preoccuparsi
(narrator) Verb eleven. To worry: {preoccuparsi}.
(repeat x3 slow) preoccuparsi
(prompt) Tell a friend: don't worry! || Non preoccuparti!
(prompt) Say it formally: don't worry! || Non si preoccupi!
(prompt) Say: my mother worries too much. || Mia madre si preoccupa troppo.
(prompt) Say to a friend: we were worried about you. Use the passato prossimo. || Ci siamo preoccupati per te.

(section) 12 — lamentarsi
(narrator) Verb twelve. To complain: {lamentarsi}.
(repeat x3 slow) lamentarsi
(prompt) Say: he always complains. || Si lamenta sempre.
(prompt) Say to two friends: you complain about everything! || Vi lamentate di tutto!
(prompt) Say: I complained to the manager. || Mi sono lamentato con il direttore.
(prompt) Say: if the service is bad, I will complain. || Se il servizio è pessimo, mi lamenterò.

(section) 13 — permettere
(narrator) Verb thirteen. To allow, to permit: {permettere}. The past participle is {permesso}.
(repeat x3 slow) permettere
(narrator) Italians often say {permesso?} when they enter someone's home. It means: may I come in?
(prompt) Say: my parents didn't let me go out. Use the imperfect. || I miei genitori non mi permettevano di uscire.
(prompt) Say: the rules don't allow it. || Le regole non lo permettono.
(prompt) Say, formally: if you allow me, I'll explain. || Se mi permette, Le spiego.
(prompt) Say: we can't afford a new car. Use {permettersi}. || Non possiamo permetterci una macchina nuova.

(section) 14 — promettere
(narrator) Verb fourteen. To promise: {promettere}. The past participle is {promesso}.
(repeat x3 slow) promettere
(prompt) Say to a friend: I promise you. || Te lo prometto.
(prompt) Say to a friend: you promised me! || Me l'hai promesso!
(prompt) Say: we promise to come back soon. || Promettiamo di tornare presto.
(prompt) Say: he promised that he would come. || Ha promesso che sarebbe venuto.

(section) 15 — risolvere
(narrator) Verb fifteen. To solve: {risolvere}. The past participle is {risolto}.
(repeat x3 slow) risolvere
(prompt) Say: we solved the problem. || Abbiamo risolto il problema.
(prompt) Say to a friend: don't worry, I'll solve it. || Non preoccuparti, lo risolvo io.
(prompt) Ask two friends: can you solve this exercise? || Riuscite a risolvere questo esercizio?
(prompt) Say: I hope they solve the situation. || Spero che risolvano la situazione.

(section) 16 — raggiungere
(narrator) Verb sixteen. To reach, or to join someone: {raggiungere}. The past participle is {raggiunto}.
(repeat x3 slow) raggiungere
(prompt) Say to two friends: I'll join you later. || Vi raggiungo più tardi.
(prompt) Say: we reached the goal. || Abbiamo raggiunto l'obiettivo.
(prompt) Say: they will reach the summit tomorrow. || Raggiungeranno la cima domani.
(prompt) Ask: how do you get to the station? Use the impersonal {si}. || Come si raggiunge la stazione?

(section) Review 2
(narrator) Quick review.
(prompt) Ask: what happened? || Che cosa è successo?
(prompt) Say: I forgot the keys. || Ho dimenticato le chiavi.
(prompt) Tell a friend: remember to call your mother! || Ricordati di chiamare la mamma!
(prompt) Say it formally: don't worry! || Non si preoccupi!
(prompt) Say: he promised that he would come. || Ha promesso che sarebbe venuto.
(prompt) Say to two friends: I'll join you later. || Vi raggiungo più tardi.

(section) 17 — ottenere
(narrator) Verb seventeen. To obtain, to get: {ottenere}. It is irregular, like {tenere}. I get is {ottengo}.
(repeat x3 slow) ottenere
(prompt) Say: I got the scholarship. || Ho ottenuto la borsa di studio.
(prompt) Say: she always gets what she wants. || Ottiene sempre quello che vuole.
(prompt) Say to a friend: you will get the visa in a month. || Otterrai il visto tra un mese.
(prompt) Say to a friend: I hope you get the job. || Spero che tu ottenga il lavoro.

(section) 18 — mantenere
(narrator) Verb eighteen. To keep, to maintain, or to support someone financially: {mantenere}. I keep is {mantengo}.
(repeat x3 slow) mantenere
(prompt) Say: I always keep my promises. || Mantengo sempre le promesse.
(prompt) Tell a friend: keep calm! || Mantieni la calma!
(prompt) Say: he kept his word. || Ha mantenuto la parola.
(prompt) Say: they support their children at university. || Mantengono i figli all'università.

(section) 19 — offrire
(narrator) Verb nineteen. To offer, or to treat someone: {offrire}. The past participle is {offerto}.
(repeat x3 slow) offrire
(prompt) Say to a friend: I'll buy you a coffee. || Ti offro un caffè.
(prompt) Say: they offered me a job. || Mi hanno offerto un lavoro.
(prompt) Ask, formally: what can I offer you? || Che cosa Le posso offrire?
(prompt) Say: tonight it's on us. || Stasera offriamo noi.

(section) 20 — chiedere
(narrator) Verb twenty. To ask: {chiedere}. The past participle is {chiesto}.
(repeat x3 slow) chiedere
(prompt) Ask a friend: can I ask you something? || Posso chiederti una cosa?
(prompt) Say: I asked for information. || Ho chiesto informazioni.
(prompt) Tell a friend: ask the waiter! || Chiedi al cameriere!
(prompt) Say to a friend: if I were you, I would ask for help. || Se fossi in te, chiederei aiuto.

(section) 21 — rispondere
(narrator) Verb twenty-one. To answer: {rispondere}. The past participle is {risposto}.
(repeat x3 slow) rispondere
(prompt) Say: he didn't answer my email. || Non ha risposto alla mia email.
(prompt) Tell a friend: answer the phone! || Rispondi al telefono!
(prompt) Say: I will answer tomorrow. || Risponderò domani.
(prompt) Ask two friends: why don't you answer? || Perché non rispondete?

(section) 22 — prestare
(narrator) Verb twenty-two. To lend: {prestare}.
(repeat x3 slow) prestare
(prompt) Ask a friend: can you lend me the car? || Mi presti la macchina?
(prompt) Say: I lent him fifty euros. || Gli ho prestato cinquanta euro.
(prompt) Say: the library lends books for a month. || La biblioteca presta i libri per un mese.
(prompt) Say: I would never lend money to them. || Non presterei mai soldi a loro.

(section) 23 — aspettarsi
(narrator) Verb twenty-three. To expect: {aspettarsi}.
(repeat x3 slow) aspettarsi
(prompt) Say: I didn't expect it. || Non me lo aspettavo.
(prompt) Ask a friend: what do you expect? || Che cosa ti aspetti?
(prompt) Say: we expect a lot from this project. || Ci aspettiamo molto da questo progetto.
(prompt) Say: they were expecting a different result. || Si aspettavano un risultato diverso.

(section) 24 — sperare
(narrator) Verb twenty-four. To hope: {sperare}.
(repeat x3 slow) sperare
(prompt) Say: I hope so. || Spero di sì.
(prompt) Say to a friend: we hope to see you soon. || Speriamo di vederti presto.
(prompt) Say to a friend: I hope you are well. || Spero che tu stia bene.
(prompt) Say: they were hoping for good weather. || Speravano nel bel tempo.

(section) 25 — provare
(narrator) Verb twenty-five. To try, or to feel: {provare}.
(repeat x3 slow) provare
(prompt) Tell a friend: try this cake! || Prova questa torta!
(prompt) Say to a friend: I tried to call you. || Ho provato a chiamarti.
(prompt) Ask a friend: what do you feel? || Che cosa provi?
(prompt) Say: we will try again tomorrow. || Proveremo di nuovo domani.

(section) Final review — verbs 1 to 25
(narrator) Final review of part A. Answer quickly.
(prompt) Say: she will manage to pass the exam. || Riuscirà a superare l'esame.
(prompt) Say to two friends: it seems to me that you are right. || Mi sembra che abbiate ragione.
(prompt) Say: we realized the problem too late. || Ci siamo resi conto del problema troppo tardi.
(prompt) Say to a friend: it's important that you decide quickly. || È importante che tu decida in fretta.
(prompt) Say to a friend: tomorrow I will explain everything to you. || Domani ti spiegherò tutto.
(prompt) Say: I'm afraid something has happened. || Ho paura che sia successo qualcosa.
(prompt) Say: we will never forget this day. || Non dimenticheremo mai questo giorno.
(prompt) Say: I complained to the manager. || Mi sono lamentato con il direttore.
(prompt) Say: we can't afford a new car. || Non possiamo permetterci una macchina nuova.
(prompt) Say: I hope they solve the situation. || Spero che risolvano la situazione.
(prompt) Say to a friend: I hope you get the job. || Spero che tu ottenga il lavoro.
(prompt) Tell a friend: keep calm! || Mantieni la calma!
(prompt) Say: they offered me a job. || Mi hanno offerto un lavoro.
(prompt) Say to a friend: if I were you, I would ask for help. || Se fossi in te, chiederei aiuto.
(prompt) Ask two friends: why don't you answer? || Perché non rispondete?
(prompt) Say: I would never lend money to them. || Non presterei mai soldi a loro.
(prompt) Say: I didn't expect it. || Non me lo aspettavo.
(prompt) Say to a friend: I hope you are well. || Spero che tu stia bene.
(prompt) Say to a friend: I tried to call you. || Ho provato a chiamarti.
(narrator) This is the end of lesson eight, part A. Tomorrow, part B: twenty-five more verbs. {A domani!}
"""

LESSONS["B"] = r"""
#####################################################################
# EVERYDAY ITALIAN — Lesson 8, Part B (B1): 50 useful verbs — verbs 26–50
# Each verb: meaning -> repeat x3 -> 4 sentences across tenses & persons.
# "you" = tu unless the prompt says formal (Lei) or plural (voi).
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson eight, part B. Fifty useful verbs, continued.
(narrator) Today you will learn verbs twenty-six to fifty. As before, repeat each verb three times. Then make sentences.
(narrator) When the prompt says "you", use {tu}. When it says "formal", use {Lei}. When it says "you, plural", use {voi}.
(narrator) First, a short warm-up with verbs from part A.
(prompt) Say: I can't manage to sleep. || Non riesco a dormire.
(prompt) Ask: what happened? || Che cosa è successo?
(prompt) Tell a friend: don't worry! || Non preoccuparti!
(prompt) Say: I hope so. || Spero di sì.
(narrator) Now, new verbs.

(section) 26 — convincere
(narrator) Verb twenty-six. To convince: {convincere}. The past participle is {convinto}.
(repeat x3 slow) convincere
(prompt) Say to a friend: you convinced me. || Mi hai convinto.
(prompt) Say: I will convince him. || Lo convincerò.
(prompt) Say: it's hard to convince them. || È difficile convincerli.
(prompt) Say to two friends: I hope to convince you. || Spero di convincervi.

(section) 27 — condividere
(narrator) Verb twenty-seven. To share: {condividere}. The past participle is {condiviso}.
(repeat x3 slow) condividere
(prompt) Say: we share a flat. || Condividiamo un appartamento.
(prompt) Say to a friend: I share your opinion. || Condivido la tua opinione.
(prompt) Say: she shared the photo. || Ha condiviso la foto.
(prompt) Tell a child: share with your brother! || Condividi con tuo fratello!

(section) 28 — evitare
(narrator) Verb twenty-eight. To avoid: {evitare}.
(repeat x3 slow) evitare
(prompt) Say: I avoid sugar. || Evito lo zucchero.
(prompt) Tell a friend: avoid the motorway on Friday. || Evita l'autostrada il venerdì.
(prompt) Say: we avoided the traffic. || Abbiamo evitato il traffico.
(prompt) Say to two friends: it's better that you avoid the centre. || È meglio che evitiate il centro.

(section) 29 — consigliare
(narrator) Verb twenty-nine. To advise, to recommend: {consigliare}.
(repeat x3 slow) consigliare
(prompt) Ask a friend: what do you advise me? || Che cosa mi consigli?
(prompt) Say: the doctor advised me to rest. || Il medico mi ha consigliato di riposare.
(prompt) Say to a friend: I would advise you to leave early. || Ti consiglierei di partire presto.
(prompt) Say: they recommended a good restaurant to me. || Mi hanno consigliato un buon ristorante.

(section) 30 — rimanere
(narrator) Verb thirty. To stay, to remain: {rimanere}. It is irregular. I stay is {rimango}. Stayed is {rimasto}, with {essere}.
(repeat x3 slow) rimanere
(prompt) Say: tonight I'm staying at home. || Stasera rimango a casa.
(prompt) Say: we stayed in Rome for a week. || Siamo rimasti a Roma una settimana.
(prompt) Ask two friends: how long will you stay? || Quanto tempo rimarrete?
(prompt) Say to a friend: I would like you to stay. || Vorrei che tu rimanessi.

(section) 31 — crescere
(narrator) Verb thirty-one. To grow, to grow up: {crescere}. The past participle is {cresciuto}, with {essere}.
(repeat x3 slow) crescere
(prompt) Say: I grew up in a small town. || Sono cresciuto in un piccolo paese.
(prompt) Say: children grow fast. || I bambini crescono in fretta.
(prompt) Say: prices will grow. || I prezzi cresceranno.
(prompt) Say to a child: how you've grown! || Come sei cresciuto!

(section) 32 — cambiare
(narrator) Verb thirty-two. To change: {cambiare}.
(repeat x3 slow) cambiare
(prompt) Say: nothing has changed. || Non è cambiato niente.
(prompt) Say: I changed my mind. || Ho cambiato idea.
(prompt) Say: things will change. || Le cose cambieranno.
(prompt) Say to two friends: if you change your mind, call me. || Se cambiate idea, chiamatemi.

(section) 33 — smettere
(narrator) Verb thirty-three. To stop, to quit: {smettere}. The past participle is {smesso}.
(repeat x3 slow) smettere
(prompt) Say: I quit smoking. || Ho smesso di fumare.
(prompt) Tell a friend: stop it! || Smettila!
(prompt) Say: it has stopped raining. || Ha smesso di piovere.
(prompt) Ask a friend: when will you stop working? || Quando smetterai di lavorare?

(section) Review 1
(narrator) Quick review.
(prompt) Say to a friend: you convinced me. || Mi hai convinto.
(prompt) Say to a friend: I share your opinion. || Condivido la tua opinione.
(prompt) Say: we avoided the traffic. || Abbiamo evitato il traffico.
(prompt) Ask a friend: what do you advise me? || Che cosa mi consigli?
(prompt) Say: we stayed in Rome for a week. || Siamo rimasti a Roma una settimana.
(prompt) Say: I grew up in a small town. || Sono cresciuto in un piccolo paese.

(section) 34 — continuare
(narrator) Verb thirty-four. To continue, to keep doing: {continuare}.
(repeat x3 slow) continuare
(prompt) Say, formally: continue straight ahead. || Continui sempre dritto.
(prompt) Say: we will continue tomorrow. || Continueremo domani.
(prompt) Say: they kept talking. Use the imperfect. || Continuavano a parlare.
(prompt) Say: I hope it doesn't keep raining. || Spero che non continui a piovere.

(section) 35 — approfittare
(narrator) Verb thirty-five. To take advantage of: {approfittare di}.
(repeat x3 slow) approfittare
(prompt) Say: let's take advantage of the nice weather. || Approfittiamo del bel tempo.
(prompt) Say: I took advantage of the sales. || Ho approfittato dei saldi.
(prompt) Tell a friend: don't take advantage of his kindness. || Non approfittare della sua gentilezza.
(prompt) Say: they will take advantage of the long weekend to go to the sea. || Approfitteranno del ponte per andare al mare.

(section) 36 — rinunciare
(narrator) Verb thirty-six. To give up, to renounce: {rinunciare a}.
(repeat x3 slow) rinunciare
(prompt) Say: I'm not giving up. || Non rinuncio.
(prompt) Say: he gave up the trip. || Ha rinunciato al viaggio.
(prompt) Tell a friend: never give up on your dreams. || Non rinunciare mai ai tuoi sogni.
(prompt) Say: we would never give up coffee. || Non rinunceremmo mai al caffè.

(section) 37 — fidarsi
(narrator) Verb thirty-seven. To trust: {fidarsi di}.
(repeat x3 slow) fidarsi
(prompt) Say to a friend: I trust you. || Mi fido di te.
(prompt) Ask a friend: do you trust him? || Ti fidi di lui?
(prompt) Tell a friend: trust me! || Fidati di me!
(prompt) Say: they didn't trust the new boss. || Non si fidavano del nuovo capo.

(section) 38 — arrabbiarsi
(narrator) Verb thirty-eight. To get angry: {arrabbiarsi}.
(repeat x3 slow) arrabbiarsi
(prompt) Tell a friend: don't get angry! || Non arrabbiarti!
(prompt) Say: she got angry with me. || Si è arrabbiata con me.
(prompt) Say: my father will get angry. || Mio padre si arrabbierà.
(prompt) Ask two friends: why are you getting angry? || Perché vi arrabbiate?

(section) 39 — divertirsi
(narrator) Verb thirty-nine. To have fun, to enjoy oneself: {divertirsi}.
(repeat x3 slow) divertirsi
(prompt) Tell two friends: have fun! || Divertitevi!
(prompt) Say: we had a great time. || Ci siamo divertiti molto.
(prompt) Ask a friend: did you have fun? || Ti sei divertito?
(prompt) Say: I hope the children have fun. || Spero che i bambini si divertano.

(section) 40 — annoiarsi
(narrator) Verb forty. To get bored: {annoiarsi}.
(repeat x3 slow) annoiarsi
(prompt) Say: I'm bored. || Mi annoio.
(prompt) Say: they got bored at the conference. || Si sono annoiati alla conferenza.
(prompt) Say to a friend: with him you will never get bored. || Con lui non ti annoierai mai.
(prompt) Say: as a child, I was often bored on Sundays. || Da bambino, la domenica mi annoiavo spesso.

(section) Review 2
(narrator) Quick review.
(prompt) Say: things will change. || Le cose cambieranno.
(prompt) Say: I quit smoking. || Ho smesso di fumare.
(prompt) Say: I hope it doesn't keep raining. || Spero che non continui a piovere.
(prompt) Say: I took advantage of the sales. || Ho approfittato dei saldi.
(prompt) Tell a friend: trust me! || Fidati di me!
(prompt) Say: she got angry with me. || Si è arrabbiata con me.
(prompt) Tell two friends: have fun! || Divertitevi!

(section) 41 — affrontare
(narrator) Verb forty-one. To face, to deal with: {affrontare}.
(repeat x3 slow) affrontare
(prompt) Say: we must face the problem. || Dobbiamo affrontare il problema.
(prompt) Say: she faced the situation with courage. || Ha affrontato la situazione con coraggio.
(prompt) Say: I will face it tomorrow. || Lo affronterò domani.
(prompt) Say to a friend: it's important that you face your fears. || È importante che tu affronti le tue paure.

(section) 42 — sostenere
(narrator) Verb forty-two. To support, or to claim: {sostenere}. It is irregular, like {tenere}.
(repeat x3 slow) sostenere
(prompt) Say: my family always supports me. || La mia famiglia mi sostiene sempre.
(prompt) Say: we support the project. || Sosteniamo il progetto.
(prompt) Say: he claims that it's true. || Sostiene che sia vero.
(prompt) Say to a friend: I will support you. || Ti sosterrò.

(section) 43 — proporre
(narrator) Verb forty-three. To propose, to suggest: {proporre}. It is irregular. I propose is {propongo}. Proposed is {proposto}.
(repeat x3 slow) proporre
(prompt) Say: I propose a toast! || Propongo un brindisi!
(prompt) Ask two friends: what do you suggest? || Che cosa proponete?
(prompt) Say: he proposed a solution. || Ha proposto una soluzione.
(prompt) Say: they would propose a new date. || Proporrebbero una nuova data.

(section) 44 — discutere
(narrator) Verb forty-four. To discuss, or to argue: {discutere}. The past participle is {discusso}.
(repeat x3 slow) discutere
(prompt) Say: we discussed the project. || Abbiamo discusso il progetto.
(prompt) Say: they always argue. || Discutono sempre.
(prompt) Say: let's discuss it tomorrow. || Ne discutiamo domani.
(prompt) Say to a friend: I don't want to argue with you. || Non voglio discutere con te.

(section) 45 — rompere
(narrator) Verb forty-five. To break: {rompere}. The past participle is {rotto}.
(repeat x3 slow) rompere
(prompt) Say: I broke my phone. || Ho rotto il telefono.
(prompt) Tell a friend: careful, you'll break it! || Attento, lo rompi!
(prompt) Say: the washing machine has broken. Use {rompersi}. || La lavatrice si è rotta.
(prompt) Ask: who broke the window? || Chi ha rotto la finestra?

(section) 46 — perdere
(narrator) Verb forty-six. To lose, to miss, or to waste: {perdere}. The past participle is {perso}.
(repeat x3 slow) perdere
(prompt) Say: I lost my wallet. || Ho perso il portafoglio.
(prompt) Tell a friend: don't waste time! || Non perdere tempo!
(prompt) Say: our team always loses. || La nostra squadra perde sempre.
(prompt) Say: if we don't hurry, we will miss the train. || Se non ci sbrighiamo, perderemo il treno.

(section) 47 — vincere
(narrator) Verb forty-seven. To win: {vincere}. The past participle is {vinto}.
(repeat x3 slow) vincere
(prompt) Ask: who won? || Chi ha vinto?
(prompt) Say: we will win next time. || Vinceremo la prossima volta.
(prompt) Say: if I won the lottery, I would travel the world. || Se vincessi la lotteria, girerei il mondo.
(prompt) Say: may the best one win! || Che vinca il migliore!

(section) 48 — dipendere
(narrator) Verb forty-eight. To depend: {dipendere da}.
(repeat x3 slow) dipendere
(prompt) Say: it depends. || Dipende.
(prompt) Say to a friend: it depends on you. || Dipende da te.
(prompt) Say: it will depend on the weather. || Dipenderà dal tempo.
(prompt) Say: it didn't depend on me. Use the imperfect. || Non dipendeva da me.

(section) 49 — risparmiare
(narrator) Verb forty-nine. To save, money or time: {risparmiare}.
(repeat x3 slow) risparmiare
(prompt) Say: I'm saving to buy a car. || Risparmio per comprare una macchina.
(prompt) Say: we saved a lot of money. || Abbiamo risparmiato molti soldi.
(prompt) Say to a friend: if you take the train, you'll save time. || Se prendi il treno, risparmierai tempo.
(prompt) Say: they should save more. || Dovrebbero risparmiare di più.

(section) 50 — occuparsi
(narrator) Verb fifty. To take care of, or to deal with: {occuparsi di}.
(repeat x3 slow) occuparsi
(prompt) Say: I'll take care of it. Use {ne}. || Me ne occupo io.
(prompt) Ask: who takes care of the children? || Chi si occupa dei bambini?
(prompt) Say: she works in marketing. Use {occuparsi di}. || Si occupa di marketing.
(prompt) Say: we will take care of everything. || Ci occuperemo di tutto.

(section) Final review — verbs 26 to 50
(narrator) Final review of part B. Answer quickly.
(prompt) Say: it's hard to convince them. || È difficile convincerli.
(prompt) Say: she shared the photo. || Ha condiviso la foto.
(prompt) Say to two friends: it's better that you avoid the centre. || È meglio che evitiate il centro.
(prompt) Say to a friend: I would advise you to leave early. || Ti consiglierei di partire presto.
(prompt) Say to a friend: I would like you to stay. || Vorrei che tu rimanessi.
(prompt) Say to two friends: if you change your mind, call me. || Se cambiate idea, chiamatemi.
(prompt) Ask a friend: when will you stop working? || Quando smetterai di lavorare?
(prompt) Tell a friend: never give up on your dreams. || Non rinunciare mai ai tuoi sogni.
(prompt) Say: they didn't trust the new boss. || Non si fidavano del nuovo capo.
(prompt) Say: I hope the children have fun. || Spero che i bambini si divertano.
(prompt) Say: as a child, I was often bored on Sundays. || Da bambino, la domenica mi annoiavo spesso.
(prompt) Say to a friend: it's important that you face your fears. || È importante che tu affronti le tue paure.
(prompt) Say: he claims that it's true. || Sostiene che sia vero.
(prompt) Say: I propose a toast! || Propongo un brindisi!
(prompt) Say: let's discuss it tomorrow. || Ne discutiamo domani.
(prompt) Say: the washing machine has broken. || La lavatrice si è rotta.
(prompt) Say: if we don't hurry, we will miss the train. || Se non ci sbrighiamo, perderemo il treno.
(prompt) Say: if I won the lottery, I would travel the world. || Se vincessi la lotteria, girerei il mondo.
(prompt) Say: it will depend on the weather. || Dipenderà dal tempo.
(prompt) Say to a friend: if you take the train, you'll save time. || Se prendi il treno, risparmierai tempo.
(prompt) Say: I'll take care of it. || Me ne occupo io.
(narrator) This is the end of lesson eight. You now know fifty new verbs. Listen to both parts again this week. {Bravissimo, e a presto!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
for part, script in LESSONS.items():
    events = tts.parse(script)
    sec, counts = tts.estimate(events)
    print(f"Part {part}: estimated {sec/60:.1f} min  | {counts.get('prompt',0)} prompts | {len(tts.jobs_for(events))} unique clips")

## 5 · Build the MP3 + transcript

In [ ]:
from IPython.display import Audio, display
outputs = []
for part, script in LESSONS.items():
    name = LESSON_NAME.replace("Lesson08_", f"Lesson08{part}_")
    mp3, txt = tts.build(script, f"{name}.mp3")
    outputs += [mp3, txt]
    display(Audio(mp3))
try:
    from google.colab import files
    for f in outputs:
        files.download(f)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
